# SQL - Window Functions

In [ ]:
import pandas as pd
import numpy as np

In [22]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

pl_customers = pl.read_csv('data/sales_customers.csv')
pl_employees = pl.read_csv('data/sales_employees.csv')
pl_orders = pl.read_csv('data/sales_orders.csv')
pl_orderarchive = pl.read_csv('data/sales_ordersarchive.csv')
pl_products = pl.read_csv('data/sales_products.csv')

### SQL TASK

#### Find the total sales across all orders

```SQL
SELECT
    sum(sales)
FROM sales.orders;
```


In [10]:
df_ts = df_orders.copy()

respuesta = pd.DataFrame({'total_sales': [df_ts['sales'].sum()]})

respuesta

,total_sales
0,380


### SQL TASK

#### Find the total sales for each product

```SQL
SELECT
    productid,
    sum(sales)
FROM sales.orders
GROUP BY productid;
```

In [16]:
df_ts = df_orders.copy()

respuesta = df_ts.groupby('productid')['sales'].sum().reset_index()

respuesta = respuesta.rename(columns={'sales':'total_sales'})

respuesta

,productid,total_sales
0,101,140
1,102,105
2,104,75
3,105,60


### SQL TASK

#### Find the total sales for each product, additionally provide details such order id & order date

```SQL
SELECT
    orderid,
    productid,
    orderdate,
    sum(sales)
FROM sales.orders
GROUP BY orders.orderid, orderdate, productid;
```

In [17]:
df_ts = df_orders.copy()

respuesta = df_ts.groupby(['orderid','orderdate','productid'])['sales'].sum().reset_index()

respuesta

,orderid,orderdate,productid,sales
0,1,2025-01-01,101,10
1,2,2025-01-05,102,15
2,3,2025-01-10,101,20
3,4,2025-01-20,105,60
4,5,2025-02-01,104,25
5,6,2025-02-05,104,50
6,7,2025-02-15,102,30
7,8,2025-02-18,101,90
8,9,2025-03-10,101,20
9,10,2025-03-15,102,60


```SQL
SELECT
    orderid,
    orderdate,
    productid,
    SUM(sales) OVER(PARTITION BY productid) TotalSalesByProducts
FROM sales.orders;
```

In [37]:
df_ts = df_orders.copy()

df_ts['TotalSalesByProducts'] = df_ts.groupby('productid')['sales'].transform('sum')

resultado = df_ts[['orderid','orderdate','productid','TotalSalesByProducts']]

resultado = resultado.sort_values('productid')

resultado

,orderid,orderdate,productid,TotalSalesByProducts
0,1,2025-01-01,101,140
2,3,2025-01-10,101,140
7,8,2025-02-18,101,140
8,9,2025-03-10,101,140
6,7,2025-02-15,102,105
1,2,2025-01-05,102,105
9,10,2025-03-15,102,105
5,6,2025-02-05,104,75
4,5,2025-02-01,104,75
3,4,2025-01-20,105,60


In [40]:
import polars as pl

resultado = pl_orders.select([
    pl.col('orderid'),
    pl.col('orderdate'),
    pl.col('productid'),
    pl.col('sales').sum().over('productid').alias('TotalSalesByProduct')
])

resultado = resultado.sort('productid')

resultado

orderid,orderdate,productid,TotalSalesByProduct
i64,str,i64,i64
1,"""2025-01-01""",101,140
3,"""2025-01-10""",101,140
8,"""2025-02-18""",101,140
9,"""2025-03-10""",101,140
2,"""2025-01-05""",102,105
7,"""2025-02-15""",102,105
10,"""2025-03-15""",102,105
5,"""2025-02-01""",104,75
6,"""2025-02-05""",104,75


### WINDOW FUNCTIONS

![window_functions](Pictures/window_functions.png)

### OVER CLAUSE

- Tells SQL that the function used is a window fucntions.
- It defines a window or suber of data


### PARTITION BY

#### Divides the result set into partitions (Windows)

### SQL TASK

#### Find the total sales across all ordwers additionally provide details such order id & order date

```SQL
SELECT
    orderid,
    orderdate,
    productid,
    SUM(sales) OVER() TotalSalesByProducts
FROM sales.orders;
```

In [27]:
df_ts = df_orders.copy()

total_global = df_ts['sales'].sum()

df_ts['GranTotalVeantas'] = total_global

resultado = df_ts[['orderid','orderdate','productid','GranTotalVeantas']]

resultado

,orderid,orderdate,productid,GranTotalVeantas
0,1,2025-01-01,101,380
1,2,2025-01-05,102,380
2,3,2025-01-10,101,380
3,4,2025-01-20,105,380
4,5,2025-02-01,104,380
5,6,2025-02-05,104,380
6,7,2025-02-15,102,380
7,8,2025-02-18,101,380
8,9,2025-03-10,101,380
9,10,2025-03-15,102,380


In [35]:
pl_ts = pl_orders

resultado = pl_ts.select([
    pl.col('orderid'),
    pl.col('orderdate'),
    pl.col('productid'),
    pl.col('sales').sum().over(pl.lit(1)).alias('GranTotalVentas')
])

resultado

orderid,orderdate,productid,GranTotalVentas
i64,str,i64,i64
1,"""2025-01-01""",101,380
2,"""2025-01-05""",102,380
3,"""2025-01-10""",101,380
4,"""2025-01-20""",105,380
5,"""2025-02-01""",104,380
6,"""2025-02-05""",104,380
7,"""2025-02-15""",102,380
8,"""2025-02-18""",101,380
9,"""2025-03-10""",101,380


- Find the total sales across all orders
- Find the total sales for each product
- Additionally provide details such order ID, order date

```SQL
SELECT
    orderid,
    orderdate,
    productid,
    sales,
    SUM(sales) OVER() TotalSalesProduct,
    SUM(sales) OVER(PARTITION BY productid) TotalSalesByProduct
FROM sales.orders;
```

In [49]:
df_ts = df_orders.copy()

df_ts['TotalSalesProduct'] = df_ts['sales'].sum()
df_ts['TotalSalesByProduct'] = df_ts.groupby('productid')['sales'].transform('sum')

resultado = df_ts[['orderid','orderdate','productid','sales','TotalSalesProduct','TotalSalesByProduct']]

resultado = resultado.sort_values('productid')

resultado


,orderid,orderdate,productid,sales,TotalSalesProduct,TotalSalesByProduct
0,1,2025-01-01,101,10,380,140
2,3,2025-01-10,101,20,380,140
7,8,2025-02-18,101,90,380,140
8,9,2025-03-10,101,20,380,140
6,7,2025-02-15,102,30,380,105
1,2,2025-01-05,102,15,380,105
9,10,2025-03-15,102,60,380,105
5,6,2025-02-05,104,50,380,75
4,5,2025-02-01,104,25,380,75
3,4,2025-01-20,105,60,380,60


In [50]:
pl_ts = pl_orders

resultado = pl_ts.select([
    pl.col('orderid'),
    pl.col('orderdate'),
    pl.col('productid'),
    pl.col('sales'),
    pl.col('sales').sum().over(pl.lit(1)).alias('TotalSalesProduct'),
    pl.col('sales').sum().over('productid').alias('TotalSalesByProduct')
])

resultado = resultado.sort('productid')

resultado

orderid,orderdate,productid,sales,TotalSalesProduct,TotalSalesByProduct
i64,str,i64,i64,i64,i64
1,"""2025-01-01""",101,10,380,140
3,"""2025-01-10""",101,20,380,140
8,"""2025-02-18""",101,90,380,140
9,"""2025-03-10""",101,20,380,140
2,"""2025-01-05""",102,15,380,105
7,"""2025-02-15""",102,30,380,105
10,"""2025-03-15""",102,60,380,105
5,"""2025-02-01""",104,25,380,75
6,"""2025-02-05""",104,50,380,75
